#EN este apartado se limpiaran los datos de datos faltantes y nulos. Haremos un documento por separado para cada uno de los grupos.
Grupo 0: CLuster categoria 0, paises Ricos UE.
Grupo 1: Cluster categoria 1, paises media UE.
Grupo 2: Cluster categoria 2, paises pobres UE.

In [1]:
import pandas as pd

In [2]:
df_ricos = pd.read_csv('C:/Users/Josue/4GA.DataScience/data/raw/Grupo0.csv')

In [3]:
df_ricos.head(2)

,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,...,trstlgl,trstplc,trstplt,trstprl,trstprt,trstsci,imbgeco,imwbcnt,happy,rlgdgr
0,ESS1e06_7,1,6.7,23.11.2023,4,BE,1.0,0.865363,0.44784,0.387544,...,9,8,8,10,NaN,NaN,99,99,99,99
1,ESS1e06_7,1,6.7,23.11.2023,9,BE,1.0,1.324481,0.44784,0.593155,...,8,9,6,7,NaN,NaN,6,4,8,6


In [4]:
#Eliminamos columnas del df que no interesen
columnas_a_eliminar = ['name', 'essround', 'edition', 'proddate', 'idno', 'dweight', 'pspwght', 'pweight', 'anweight', 'prob', 'stratum', 'psu']
df_ricos.drop(columnas_a_eliminar, axis=1, inplace=True)

In [5]:
#Obtenemos una lista con los nombres de las columnas restantes.
nombres_columnas = df_ricos.columns.tolist()
print(nombres_columnas)

['cntry', 'pplfair', 'pplhlp', 'ppltrst', 'lawobey', 'lrscale', 'polintr', 'stfdem', 'stfeco', 'stfgov', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstsci', 'imbgeco', 'imwbcnt', 'happy', 'rlgdgr']


In [6]:
#Obtenemos una lista con los paises de las muestras de las filas.
valores_unicos_cntry = df_ricos['cntry'].unique()
print(valores_unicos_cntry)

['BE' 'DE' 'NL']


In [7]:
#Guardamos en un dataframe nuestra tabla para asignar grupo y valores clave de la columna cntry.
import sqlite3
import pandas as pd


db_path = 'C:/Users/Josue/4GA.DataScience/src/EcoUE.db'
try:
    conn = sqlite3.connect(db_path)
    query = "SELECT * FROM ecoeu"
    df_ecoeu = pd.read_sql_query(query, conn)
    print(df_ecoeu)
except sqlite3.Error as e:
    print(f"Error al conectar o consultar la base de datos: {e}")

finally:
    
    if conn:
        conn.close()

           cntry       PIB  Inflation    sma       cntrycat  \
0       Alemania   54343.2        5.9  60867           Rico   
1        Austria   56033.6        7.8  57082           Rico   
2        Bélgica   54700.9        4.0  59285           Rico   
3         Chipre   36551.4        3.5  26689  Media Europea   
4        Croacia   21865.5        7.9  17714  Media Europea   
5      Eslovenia   32610.1        7.4  26667  Media Europea   
6         España   33509.0        3.5  30237  Media Europea   
7        Estonia   30133.3        9.2  21595  Media Europea   
8      Finlandia   52925.7        6.3  53310           Rico   
9        Francia   44690.9        4.9  43438  Media Europea   
10        Grecia   23400.7        3.5  23536  Media Europea   
11       Irlanda  103887.8        6.3  59899           Rico   
12        Italia   39003.3        5.6  33492  Media Europea   
13       Letonia   22502.8        8.9  18559  Media Europea   
14      Lituania   27786.0        9.1  23409  Media Eur

In [8]:
cntrymap= {
    'BE':'Bélgica',
    'NL':'Países Bajos',
    'DE':'Alemania',
    
}
df_ricos['cntry'] = df_ricos['cntry'].map(cntrymap)
df_ricos = pd.merge(df_ricos, df_ecoeu[['cntry', 'cntrycat_factorizado']], on='cntry', how='left')

In [9]:
print(df_ricos[['cntry', 'cntrycat_factorizado']])

              cntry  cntrycat_factorizado
0           Bélgica                     0
1           Bélgica                     0
2           Bélgica                     0
3           Bélgica                     0
4           Bélgica                     0
...             ...                   ...
75909  Países Bajos                     0
75910  Países Bajos                     0
75911  Países Bajos                     0
75912  Países Bajos                     0
75913  Países Bajos                     0

[75914 rows x 2 columns]


In [10]:
df_ricos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75914 entries, 0 to 75913
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 75914 non-null  object 
 1   pplfair               75914 non-null  int64  
 2   pplhlp                75914 non-null  int64  
 3   ppltrst               75914 non-null  int64  
 4   lawobey               7182 non-null   float64
 5   lrscale               75914 non-null  int64  
 6   polintr               75914 non-null  int64  
 7   stfdem                75914 non-null  int64  
 8   stfeco                75914 non-null  int64  
 9   stfgov                75914 non-null  int64  
 10  trstlgl               75914 non-null  int64  
 11  trstplc               75914 non-null  int64  
 12  trstplt               75914 non-null  int64  
 13  trstprl               75914 non-null  int64  
 14  trstprt               68732 non-null  float64
 15  trstsci            

In [11]:
import numpy as np, random
df_ricos['lawobey'] = np.nan

In [ ]:
import pandas as pd
import numpy as np

def marcar_atipicos_ordinales_nan_contador(df):
    """
    Marca los valores atípicos en columnas ordinales con NaN y cuenta los atípicos en 'contadornegativo'.


    """

    df_modificado = df.copy()
    columnas_numericas = df_modificado.select_dtypes(include=np.number).columns
    df_modificado['contadornegativo'] = 0  # Inicializa la columna 'contadornegativo'

    for columna in columnas_numericas:
        max_valor = df_modificado[columna].max()

        if max_valor == 9:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado.loc[df_modificado[columna].isin([6, 7, 8, 9])].index

            # Marca los atípicos con NaN
            df_modificado.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado.loc[indices_atipicos, 'contadornegativo'] += 1

        elif max_valor == 5:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado.loc[df_modificado[columna].isin([0, 5])].index

            # Marca los atípicos con NaN
            df_modificado.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado.loc[indices_atipicos, 'contadornegativo'] += 1

    return df_modificado

# Ejemplo de uso:
# Supongamos que tienes tu DataFrame llamado df
df_ricos = marcar_atipicos_ordinales_nan_contador(df_ricos)

# Imprime la forma del DataFrame original y del DataFrame modificado
print(f"Forma del DataFrame original: {df_ricos.shape}")
print(f"Forma del DataFrame modificado: {df_ricos.shape}")

# Imprime la cantidad de valores NaN por columna en el DataFrame modificado
print(df_ricos.isnull().sum())

# Imprime la columna 'contadornegativo'
print(df_ricos['contadornegativo'].head(10)) #imprime los primeros 10 valores

Forma del DataFrame original: (75914, 22)
Forma del DataFrame modificado: (75914, 22)
cntry                       0
pplfair                     0
pplhlp                      0
ppltrst                     0
lawobey                 75914
lrscale                     0
polintr                    86
stfdem                      0
stfeco                      0
stfgov                      0
trstlgl                     0
trstplc                     0
trstplt                     0
trstprl                     0
trstprt                  7182
trstsci                 64378
imbgeco                     0
imwbcnt                     0
happy                       0
rlgdgr                      0
cntrycat_factorizado        0
contadornegativo            0
dtype: int64
0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: contadornegativo, dtype: int64


In [ ]:
import pandas as pd
import numpy as np

def marcar_atipicos_ordinales_nan_contador_10_acumulativo_df2(df):
    """
    Marca los valores atípicos 66, 77, 88, 99 en columnas ordinales con NaN y acumula los atípicos en 'contadornegativo' .

    """

    df_modificado2 = df.copy()
    columnas_numericas = df_modificado2.select_dtypes(include=np.number).columns

    # Asegurarse de que 'contadornegativo' existe, si no, inicializarla.
    if 'contadornegativo' not in df_modificado2.columns:
        df_modificado2['contadornegativo'] = 0

    for columna in columnas_numericas:
        max_valor = df_modificado2[columna].max()

        if max_valor == 99:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado2.loc[df_modificado2[columna].isin([66, 77, 88, 99])].index

            # Marca los atípicos con NaN
            df_modificado2.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado2.loc[indices_atipicos, 'contadornegativo'] += 1

        elif max_valor == 10:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado2.loc[df_modificado2[columna].isin([0, 10])].index

            # Marca los atípicos con NaN
            df_modificado2.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado2.loc[indices_atipicos, 'contadornegativo'] += 1

    return df_modificado2

# Ejemplo de uso:
# Supongamos que tienes tu DataFrame llamado df_modificado2
df_ricos = marcar_atipicos_ordinales_nan_contador_10_acumulativo_df2(df_ricos)

# Imprime la forma del DataFrame original y del DataFrame modificado
print(f"Forma del DataFrame original: {df_ricos.shape}")


# Imprime la cantidad de valores NaN por columna en el DataFrame modificado
print(df_ricos.isnull().sum())

# Imprime la columna 'contadornegativo'
print(df_ricos['contadornegativo'].head(10))

Forma del DataFrame original: (75914, 22)
Forma del DataFrame modificado: (75914, 22)
cntry                       0
pplfair                   248
pplhlp                    205
ppltrst                   142
lawobey                 75914
lrscale                  4488
polintr                    86
stfdem                   1273
stfeco                   1065
stfgov                   1739
trstlgl                   842
trstplc                   381
trstplt                   752
trstprl                  1252
trstprt                  8050
trstsci                 64560
imbgeco                  1669
imwbcnt                  1475
happy                     247
rlgdgr                    414
cntrycat_factorizado        0
contadornegativo            0
dtype: int64
0    8
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    5
9    0
Name: contadornegativo, dtype: int64


In [14]:
import pandas as pd
import numpy as np

databel = {
    'lawobey': {
        1: 0.344,
        2: 0.398,
        3: 0.137,
        4: 0.106,
        5: 0.016,
    },
    'trstsci': {
        0: 0.009,
        1: 0.009,
        2: 0.013,
        3: 0.016,
        4: 0.027,
        5: 0.083,
        6: 0.096,
        7: 0.207,
        8: 0.317,
        9: 0.159,
        10: 0.061,
    }
}

def rngpond(df, dataspa, cntry_value, missing_threshold):

    df_cntry = df[df['cntry'] == cntry_value].copy()  

    for columna, pesos in dataspa.items():
        if columna in df_cntry.columns and df_cntry[columna].isnull().any():
            valores = np.array(list(pesos.keys()))
            pesos_ponderados = np.array(list(pesos.values()))

            
            suma_pesos = np.sum(pesos_ponderados)
            pesos_normalizados = pesos_ponderados / suma_pesos

            for index, row in df_cntry[df_cntry[columna].isnull()].iterrows():
                
                valor_aleatorio = np.random.choice(valores, p=pesos_normalizados)

                
                df_cntry.loc[index, columna] = valor_aleatorio

    
    for columna in df_cntry.columns:
        if df_cntry[columna].isnull().any():
            missing_percentage = df_cntry[columna].isnull().sum() / len(df_cntry)
            if missing_percentage < missing_threshold:
                mode_value = df_cntry[columna].mode()[0]
                df_cntry[columna].fillna(mode_value, inplace=True)

    
    df.loc[df['cntry'] == cntry_value] = df_cntry 


rngpond(df_ricos, databel, 'Bélgica', missing_threshold=0.2)
print(df_ricos[df_ricos['cntry'] == 'Bélgica'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


C:\Users\Josue\AppData\Local\Temp\ipykernel_9676\464865049.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cntry[columna].fillna(mode_value, inplace=True)


In [15]:
datahol = {
    'lawobey': {
        1: 0.14,
        2: 0.413,
        3: 0.225,
        4: 0.209,
        5: 0.014,
    },
    'trstsci': {
        0: 0.004,
        1: 0.002,
        2: 0.009,
        3: 0.01,
        4: 0.015,
        5: 0.054,
        6: 0.085,
        7: 0.234,
        8: 0.34,
        9: 0.197,
        10: 0.048,
    }
}
rngpond(df_ricos, datahol, 'Países Bajos', missing_threshold=0.2)
print(df_ricos[df_ricos['cntry'] == 'Países Bajos'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


C:\Users\Josue\AppData\Local\Temp\ipykernel_9676\464865049.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cntry[columna].fillna(mode_value, inplace=True)


In [16]:
datager = {
    'lawobey': {
        1: 0.192,
        2: 0.532,
        3: 0.149,
        4: 0.115,
        5: 0.012,
    },
    'trstsci': {
        0: 0.017,
        1: 0.009,
        2: 0.024,
        3: 0.037,
        4: 0.043,
        5: 0.119,
        6: 0.095,
        7: 0.16,
        8: 0.228,
        9: 0.156,
        10: 0.114,
    }
}
rngpond(df_ricos, datager, 'Alemania', missing_threshold=0.2)
print(df_ricos[df_ricos['cntry'] == 'Alemania'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


C:\Users\Josue\AppData\Local\Temp\ipykernel_9676\464865049.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cntry[columna].fillna(mode_value, inplace=True)


In [17]:
df_ricos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75914 entries, 0 to 75913
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 75914 non-null  object 
 1   pplfair               75914 non-null  float64
 2   pplhlp                75914 non-null  float64
 3   ppltrst               75914 non-null  float64
 4   lawobey               75914 non-null  float64
 5   lrscale               75914 non-null  float64
 6   polintr               75914 non-null  float64
 7   stfdem                75914 non-null  float64
 8   stfeco                75914 non-null  float64
 9   stfgov                75914 non-null  float64
 10  trstlgl               75914 non-null  float64
 11  trstplc               75914 non-null  float64
 12  trstplt               75914 non-null  float64
 13  trstprl               75914 non-null  float64
 14  trstprt               75914 non-null  float64
 15  trstsci            

In [18]:
import pandas as pd
import numpy as np

def contar_neutros_ponderado(df):
    """
    Crea una columna 'ContadorNeutro' con neutralidad ponderada para diferentes escalas.

    Args:
        df (pd.DataFrame): El DataFrame.

    Returns:
        pd.DataFrame: El DataFrame con la nueva columna 'ContadorNeutro'.
    """

    df_modificado = df.copy()
    df_modificado['ContadorNeutro'] = 0.0

    for columna in df_modificado.select_dtypes(include=np.number).columns:
        valores_unicos = df_modificado[columna].dropna().unique()
        num_categorias = len(valores_unicos)

        if num_categorias > 2:
            if num_categorias == 3:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 0.25, 0)
            elif num_categorias == 4:
                cuartiles = np.quantile(valores_unicos, [0.25, 0.75])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= cuartiles[0]) & (df_modificado[columna] <= cuartiles[1]), 0.25, 0)
            elif num_categorias == 5:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 0.5, 0)
            elif num_categorias == 6:
                sextiles = np.quantile(valores_unicos, [1/3, 2/3])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= sextiles[0]) & (df_modificado[columna] <= sextiles[1]), 0.33, 0)
            elif num_categorias == 7:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 1.0, 0)
            elif num_categorias == 8:
                cuartiles_centrales = np.quantile(valores_unicos, [0.375, 0.625])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= cuartiles_centrales[0]) & (df_modificado[columna] <= cuartiles_centrales[1]), 0.66, 0)
            elif num_categorias == 9:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 1.25, 0)
            elif num_categorias == 10:
                quintiles_centrales = np.quantile(valores_unicos, [0.4, 0.6])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= quintiles_centrales[0]) & (df_modificado[columna] <= quintiles_centrales[1]), 1.0, 0)
            elif num_categorias == 11:
                sextil_central = np.quantile(valores_unicos, 0.5)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == sextil_central, 1.5, 0)

    return df_modificado

# Ejemplo de uso:
df_ricos = contar_neutros_ponderado(df_ricos)

In [19]:
import pandas as pd
import numpy as np

def contar_extremos_ponderado_igual_neutro(df):
    """
    Crea una columna 'ContadorPositivo' con extremos ponderados iguales a los neutros.

    Args:
        df (pd.DataFrame): El DataFrame.

    Returns:
        pd.DataFrame: El DataFrame con la nueva columna 'ContadorPositivo'.
    """

    df_modificado = df.copy()
    df_modificado['ContadorPositivo'] = 0.0

    for columna in df_modificado.select_dtypes(include=np.number).columns:
        valores_unicos = df_modificado[columna].dropna().unique()
        num_categorias = len(valores_unicos)

        if num_categorias > 2:
            valor_minimo = np.min(valores_unicos)
            valor_maximo = np.max(valores_unicos)

            if num_categorias == 3:
                ponderacion_extremo = 0.25
            elif num_categorias == 4:
                ponderacion_extremo = 0.25
            elif num_categorias == 5:
                ponderacion_extremo = 0.5
            elif num_categorias == 6:
                ponderacion_extremo = 0.33
            elif num_categorias == 7:
                ponderacion_extremo = 1.0
            elif num_categorias == 8:
                ponderacion_extremo = 0.66
            elif num_categorias == 9:
                ponderacion_extremo = 1.25
            elif num_categorias == 10:
                ponderacion_extremo = 1.0
            elif num_categorias == 11:
                ponderacion_extremo = 1.5
            else:
                ponderacion_extremo = 0  # Valor predeterminado para otras categorias

            df_modificado['ContadorPositivo'] += np.where((df_modificado[columna] == valor_minimo) | (df_modificado[columna] == valor_maximo), ponderacion_extremo, 0)

    return df_modificado

# Ejemplo de uso:
df_ricos= contar_extremos_ponderado_igual_neutro(df_ricos)

In [20]:
df_ricos.rename(columns={'lawobey': 'lw_pnd', 'trstsci': 'trtsci_pnd','cntrycat_factorizado': 'cntgrp_fc',}, inplace=True)

In [21]:
df_ricos.to_csv('C:/Users/Josue/4GA.DataScience/data/raw/UEricG0.csv',index=False)

In [29]:
%reset -f